# Week 10 Lab — Panel Data Estimators

**MANG2074 Financial Econometrics 1**

**Objectives**

- Estimate pooled OLS, fixed effects and random effects with `linearmodels`.
- Verify the within-transformation/LSDV equivalence numerically.
- Conduct the Hausman test and choose between FE and RE.
- Apply entity-clustered standard errors.

**Data**

- `../data/panelx.csv` — `firm_ident`, `return`, `beta`, `year`: 1,734 firms, 1996–2006.
- `../data/panel_wage.csv` — Cornwell–Rupert wage panel: 595 workers (`id`) × 7 years (`t`); `lwage`, `exp`, `exp2`, `wks`, `ms`, `union`, etc.


## Setup

Load both panels and set the (entity, time) MultiIndex that `linearmodels` requires.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels import PooledOLS, PanelOLS, RandomEffects
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- firm panel ---
firms = pd.read_csv('../data/panelx.csv')
firms = firms.rename(columns={'return': 'ret'})   # 'return' is a Python keyword
firms = firms.dropna()
firms = firms.set_index(['firm_ident', 'year'])   # MultiIndex: entity, time
print(firms.head())

# --- wage panel ---
wage = pd.read_csv('../data/panel_wage.csv')
wage = wage.set_index(['id', 't'])
print(wage[['lwage', 'exp', 'exp2', 'wks', 'ms', 'union']].describe().round(3))

## Task 1 — Pooled OLS on the firm panel

Estimate `ret ~ 1 + beta` by `PooledOLS`. Does beta carry a positive premium, as the CAPM/SML would suggest? What assumption does pooling make about firm heterogeneity?

In [ ]:
pooled = PooledOLS.from_formula('ret ~ 1 + beta', firms).fit()
# YOUR CODE HERE: print(pooled)


## Task 2 — Fixed effects

Estimate the same equation with firm fixed effects (`'ret ~ 1 + beta + EntityEffects'`, `PanelOLS`). Compare the beta coefficient with pooled OLS. What changed and why?

In [ ]:
# YOUR CODE HERE: fe = PanelOLS.from_formula(...).fit(); print(fe)


## Task 3 — FE is just demeaning: verify

Implement the within transformation by hand: subtract entity means from `ret` and `beta` (`groupby(level=0).transform("mean")`), run plain OLS on the demeaned data (no constant), and confirm the slope equals the `PanelOLS` estimate.

In [ ]:
ret_dm = firms['ret'] - firms.groupby(level=0)['ret'].transform('mean')
# YOUR CODE HERE: beta_dm, then sm.OLS(ret_dm, beta_dm).fit() and compare slopes


## Task 4 — Random effects and the Hausman test

Estimate `RandomEffects`, then compute the Hausman statistic manually for the single slope:

$$H = \frac{(\hat\beta_{FE} - \hat\beta_{RE})^2}{\widehat{Var}(\hat\beta_{FE}) - \widehat{Var}(\hat\beta_{RE})} \sim \chi^2(1)$$

State the null and the decision rule, and conclude: FE or RE for this dataset?

In [ ]:
re = RandomEffects.from_formula('ret ~ 1 + beta', firms).fit()
# YOUR CODE HERE: Hausman statistic and p-value (stats.chi2.sf)


## Task 5 — Cluster-robust standard errors

Re-fit the FE model with `cov_type='clustered', cluster_entity=True`. How much do the SEs change relative to the default? Why are clustered SEs the norm in finance panels?

In [ ]:
# YOUR CODE HERE


## Task 6 — A wage equation

On the wage panel estimate pooled OLS, FE and RE for

`lwage ~ 1 + exp + exp2 + wks + ms + union`

Collect the coefficients in one comparison table. Which coefficients flip sign between pooled and FE? What does that tell you about worker heterogeneity?

In [ ]:
fw = 'lwage ~ 1 + exp + exp2 + wks + ms + union'
# YOUR CODE HERE: three models + comparison table (hint: pd.DataFrame({'pooled': ..., 'FE': ..., 'RE': ...}))


## Task 7 — Hausman test for the wage equation

Compute the Hausman statistic for the full slope vector (5 slopes):

$$H = (\hat\beta_{FE}-\hat\beta_{RE})'[\widehat{V}_{FE}-\widehat{V}_{RE}]^{-1}(\hat\beta_{FE}-\hat\beta_{RE}) \sim \chi^2(5)$$

(Use `.params` and `.cov`, dropping the intercept.) Conclusion?

In [ ]:
# YOUR CODE HERE


## Task 8 — Write-up

5–8 sentences: summarise the beta–return finding (and its relation to the CAPM), the wage-equation finding, and your estimator choices with justification.

*YOUR ANSWER HERE*